Part 1 — Existence check (near-on-time timetable exists)

Metric: q5 of run-level average delay per train (the 5% quantile across the 200 runs).
Threshold: q5 ≤ 15 seconds.
Meaning: At least 10 out of 200 drafts achieve ≤15s average delay per train. A workable timetable exists; we’re not demanding perfection.
Part 2 — Flexibility check (not a “one-lucky-plan”)
Pick one of these simple, explainable options:

Option A: Pass rate
Metric: PR = fraction of runs with average delay per train ≤ 15s.
Threshold: PR ≥ 20% (40 of 200 runs).
Why: Ensures there are many workable timetables, not just a one-off. You can tune 20% to 10–30% based on calibration.
Option B: Repairability (bounded retiming)
Allow up to ±15s planned shift per train; check if retiming eliminates operational waiting.
Metric: RP = fraction of runs that become “no waiting” with max |Δt| ≤ 15s.
Threshold: RP ≥ 50% and 95th-percentile max |Δt| ≤ 15s.
Why: Shows most drafts can be made feasible with small, realistic tweaks—strong flexibility for chaining multiple blocks.
Capacity definition

Capacity (N*) is the highest train volume for which both the existence check (q5 ≤ 15s) and the flexibility check (Option A or B) pass.
Report N* together with the flexibility score you used (PR or RP), so stakeholders see not just “how many trains” but “how flexible at that level.”

In [11]:
import pandas as pd
from pathlib import Path

import numpy as np

In [6]:
filer = Path(r"/Users/nicolasdulex/devsbb/GZB_analysis")
run_id = "output_20260219_it5_n5"
OUTPUT_ROOT = filer / run_id

In [13]:
df = pd.read_csv(OUTPUT_ROOT / "output_run_summary.csv")

In [14]:
pd

<module 'pandas' from '/Users/nicolasdulex/devsbb/GZB_analysis/.venv/lib/python3.14/site-packages/pandas/__init__.py'>

In [15]:
#print the columns of the dataframe and type 
print(pd.columns)
print(pd.dtypes)

AttributeError: module 'pandas' has no attribute 'columns'

In [ ]:


def compute_pass_rate(
    df: pd.DataFrame,
    group_cols=('use_case','building_block','operating_mode','product_mix','flow_pattern','train_volume'),
    delay_threshold_s=15.0,
    require_no_headway_violations=False,
    require_no_stuck=True,
    exclude_no_arrivals=True
):
    """
    Compute pass rate per configuration:
      - metric per run = average delay per arrived train in the analysis window
                         = analysis_window_delay_at_destination / trains_arrived
      - a run "passes" if metric <= delay_threshold_s
        (+ optional: no headway violations, no stuck trains)
      - pass_rate = n_pass / n_evaluable (or / n_runs if exclude_no_arrivals=False)

    Parameters
    ----------
    df : DataFrame with columns listed by the user.
    group_cols : columns that define a configuration (tuple/list).
    delay_threshold_s : per-train delay threshold (seconds).
    require_no_headway_violations : if True, require 0s headway violations in window.
    require_no_stuck : if True, require trains_stuck == 0.
    exclude_no_arrivals : if True, runs with trains_arrived == 0 are excluded from denominator.

    Returns
    -------
    DataFrame with columns:
      group_cols..., n_runs, n_evaluable, n_pass, pass_rate
    """
    tmp = df.copy()

    # Per-run avg delay per arrived train in analysis window
    with np.errstate(divide='ignore', invalid='ignore'):
        tmp['avg_delay_s'] = tmp['analysis_window_delay_at_destination'] / tmp['trains_arrived']
    tmp.loc[~np.isfinite(tmp['avg_delay_s']), 'avg_delay_s'] = np.nan  # handle inf/-inf

    # Base pass on delay threshold
    pass_mask = tmp['avg_delay_s'] <= delay_threshold_s

    # Optional constraints
    if require_no_headway_violations:
        hv_ok = (
            (tmp['analysis_window_headway_violation_tail_to_head'] <= 0.0) &
            (tmp['analysis_window_headway_violation_head_to_head'] <= 0.0)
        )
        pass_mask &= hv_ok

    if require_no_stuck:
        pass_mask &= (tmp['trains_stuck'] == 0)

    tmp['pass'] = pass_mask

    # Define evaluable runs (those with at least one arrival) if excluding
    if exclude_no_arrivals:
        evaluable = tmp['trains_arrived'] > 0
    else:
        evaluable = pd.Series(True, index=tmp.index)

    tmp['evaluable'] = evaluable

    # Aggregate
    grp = tmp.groupby(list(group_cols), dropna=False)
    out = grp.agg(
        n_runs=('run_id', 'nunique'),
        n_evaluable=('evaluable', 'sum'),
        n_pass=('pass', 'sum')
    ).reset_index()

    # Avoid divide-by-zero; define pass_rate over evaluable runs by default
    denom = out['n_evaluable'] if exclude_no_arrivals else out['n_runs']
    out['pass_rate'] = np.where(denom > 0, out['n_pass'] / denom, np.nan)

    return out




,use_case,building_block,operating_mode,product_mix,flow_pattern,train_volume,n_runs,n_evaluable,n_pass,pass_rate
0,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,11,2,2,2,1.0
1,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,12,2,2,2,1.0
2,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,13,2,2,2,1.0
3,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,14,2,2,1,0.5
4,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,15,2,2,0,0.0
...,...,...,...,...,...,...,...,...,...,...
214,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,8,2,2,2,1.0
215,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,9,2,2,2,1.0
216,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,10,2,2,0,0.0
217,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,11,2,2,2,1.0


In [18]:
# Example usage:
res = compute_pass_rate(df,
    group_cols=['use_case','building_block','operating_mode','product_mix','flow_pattern','train_volume'],
    delay_threshold_s=15.0,
    require_no_headway_violations=False,
    require_no_stuck=True,
    exclude_no_arrivals=True
)
res

,use_case,building_block,operating_mode,product_mix,flow_pattern,train_volume,n_runs,n_evaluable,n_pass,pass_rate
0,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,11,2,2,2,1.0
1,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,12,2,2,2,1.0
2,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,13,2,2,2,1.0
3,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,14,2,2,1,0.5
4,UC_0,UC0_BB1,express_pass,EXPRESS,PASS,15,2,2,0,0.0
...,...,...,...,...,...,...,...,...,...,...
214,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,8,2,2,2,1.0
215,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,9,2,2,2,1.0
216,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,10,2,2,0,0.0
217,UC_2,UC2_BB2,transit_trunk,TRANSIT,TRUNK,11,2,2,2,1.0
